In [3]:
pip install pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib # Used for saving the trained models to files
import os # Used to check if the file exists

def train_ev_prediction_models(csv_path='ev_dataset.csv'):
    """
    This is the main function that does everything:
    1. Loads the data from the CSV.
    2. Defines which columns are inputs (features) and outputs (targets).
    3. Sets up a preprocessing pipeline to handle text and numbers.
    4. Trains three separate Linear Regression models.
    5. Saves the three trained models to disk as .pkl files.
    """
    
    # --- Step 1: Check for and Load Data ---
    if not os.path.exists(csv_path):
        print(f"Error: The file {csv_path} was not found.")
        print("Please make sure 'electric_vehicle_analytics.csv' is in the same folder as this script.")
        return None, None, None
        
    try:
        print(f"Loading data from {csv_path}...")
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"Error loading CSV file: {e}")
        return None, None, None

    try:
        # --- Step 2: Define Features (X) and Targets (y) ---
        
        # These are the columns we will use as INPUTS
        features = ['Make', 'Model', 'Year', 'Mileage_km', 'Battery_Capacity_kWh']
        
        # These are the columns we want to PREDICT
        target_health = 'Battery_Health_%'
        target_time = 'Charging_Time_hr'
        target_cost = 'Monthly_Charging_Cost_USD'

        # Check if all required columns exist in the CSV
        required_cols = features + [target_health, target_time, target_cost]
        missing_cols = [col for col in required_cols if col not in df.columns]
        
        if missing_cols:
            print(f"Error: The CSV file is missing required columns: {missing_cols}")
            return None, None, None

        # Create the X (input) and y (output) dataframes
        X = df[features]
        y_health = df[target_health]
        y_time = df[target_time]
        y_cost = df[target_cost]

        # --- Step 3: Preprocessing ---
        # We must convert text columns ('Make', 'Model') into numbers
        # so the Linear Regression model can understand them.
        
        categorical_features = ['Make', 'Model']
        numerical_features = ['Year', 'Mileage_km', 'Battery_Capacity_kWh']

        # Create a "preprocessor" that applies different steps to different columns
        preprocessor = ColumnTransformer(
            transformers=[
                # Apply 'passthrough' (no change) to numerical features
                ('num', 'passthrough', numerical_features),
                # Apply OneHotEncoder to categorical features
                # handle_unknown='ignore' prevents errors if we see a new 'Make' or 'Model'
                ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
            ])

        # --- Step 4: Create and Train Models ---
        # We create a "Pipeline" for each model. This pipeline automatically
        # runs the preprocessor (Step 3) and then runs the regression model.

        # Model 1: Predicts Battery Health
        pipeline_health = Pipeline(steps=[('preprocessor', preprocessor),
                                          ('regressor', LinearRegression())])
        
        # Model 2: Predicts Charging Time
        pipeline_time = Pipeline(steps=[('preprocessor', preprocessor),
                                        ('regressor', LinearRegression())])
        
        # Model 3: Predicts Monthly Cost
        pipeline_cost = Pipeline(steps=[('preprocessor', preprocessor),
                                        ('regressor', LinearRegression())])

        # Now, we train all three models on our data
        print("Training models...")
        pipeline_health.fit(X, y_health)
        pipeline_time.fit(X, y_time)
        pipeline_cost.fit(X, y_cost)
        print("Models trained successfully.")

        # --- Step 5: Save Models to Disk ---
        # This is the most important part. We save the trained pipelines
        # so another script (like a chatbot) can load them later.
        
        joblib.dump(pipeline_health, 'ev_health_model.pkl')
        joblib.dump(pipeline_time, 'ev_time_model.pkl')
        joblib.dump(pipeline_cost, 'ev_cost_model.pkl')
        
        print("\nModels saved to disk:")
        print("- ev_health_model.pkl (predicts Battery Health %)")
        print("- ev_time_model.pkl (predicts Charging Time hr)")
        print("- ev_cost_model.pkl (predicts Monthly Charging Cost USD)")
        
        return pipeline_health, pipeline_time, pipeline_cost

    except Exception as e:
        print(f"An unexpected error occurred during training: {e}")
        return None, None, None

# --- Example of how to use the models ---
# This code block only runs when you execute `python train_model.py`
if __name__ == "__main__":
    
    # Run the main training function
    model_health, model_time, model_cost = train_ev_prediction_models()
    
    # Check if models were trained successfully
    if model_health and model_time and model_cost:
        
        # You can also load the saved models from the files like this:
        # print("\nLoading models from disk for a test...")
        # loaded_health_model = joblib.load('ev_health_model.pkl')
        # loaded_time_model = joblib.load('ev_time_model.pkl')
        # loaded_cost_model = joblib.load('ev_cost_model.pkl')
        
        # Create some new, sample data to predict
        # This MUST be a pandas DataFrame with the *exact* same column names
        # that the model was trained on:
        # ['Make', 'Model', 'Year', 'Mileage_km', 'Battery_Capacity_kWh']
        new_data = pd.DataFrame({
            'Make': ['Nissan', 'Tesla', 'Ford'],
            'Model': ['Leaf', 'Model 3', 'Mustang Mach-E'],
            'Year': [2021, 2023, 2024],
            'Mileage_km': [50000, 15000, 5000],
            'Battery_Capacity_kWh': [40, 75, 99]
        })
        
        print("\n--- Making Predictions on New Data ---")
        
        # Use the trained models (from memory) to predict
        pred_health = model_health.predict(new_data)
        pred_time = model_time.predict(new_data)
        pred_cost = model_cost.predict(new_data)
        
        for i in range(len(new_data)):
            print(f"\nPrediction for {new_data.iloc[i]['Year']} {new_data.iloc[i]['Make']} {new_data.iloc[i]['Model']}:")
            print(f"  Predicted Battery Health: {pred_health[i]:.2f} %")
            print(f"  Predicted Charging Time:  {pred_time[i]:.2f} hours")
            print(f"  Predicted Monthly Cost:   ${pred_cost[i]:.2f}")

Loading data from ev_dataset.csv...
Training models...
Models trained successfully.

Models saved to disk:
- ev_health_model.pkl (predicts Battery Health %)
- ev_time_model.pkl (predicts Charging Time hr)
- ev_cost_model.pkl (predicts Monthly Charging Cost USD)

--- Making Predictions on New Data ---

Prediction for 2021 Nissan Leaf:
  Predicted Battery Health: 85.01 %
  Predicted Charging Time:  0.69 hours
  Predicted Monthly Cost:   $163.64

Prediction for 2023 Tesla Model 3:
  Predicted Battery Health: 84.65 %
  Predicted Charging Time:  1.23 hours
  Predicted Monthly Cost:   $49.79

Prediction for 2024 Ford Mustang Mach-E:
  Predicted Battery Health: 84.45 %
  Predicted Charging Time:  1.60 hours
  Predicted Monthly Cost:   $18.59
